In [1]:
# Imports and Raw Data

import pandas as pd
import numpy as np

macro = pd.read_parquet("../data/macro_raw.parquet")

macro.tail()

,core_cpi,fed_funds,unemployment,oil_price,m2
date,,,,,
2026-04-07,NaN,NaN,NaN,114.58,NaN
2026-04-08,NaN,NaN,NaN,96.17,NaN
2026-04-09,NaN,NaN,NaN,99.62,NaN
2026-04-10,NaN,NaN,NaN,98.34,NaN
2026-04-13,NaN,NaN,NaN,100.72,NaN


In [2]:
# FORWARD FILL MONTHLY DATA TO DAILY FREQUENCY

macro_ffill = macro.ffill()
macro_ffill.tail()

"""
We're forwarding filling because Kalshi events resolve daily probability, and the trading engine needs daily signals.
"""

"\nWe're forwarding filling because Kalshi events resolve daily probability, and the trading engine needs daily signals.\n"

In [3]:
# CREATING BASIC INFLATION FEATURES

macro_ffill["core_cpi_yoy"] = macro_ffill["core_cpi"].pct_change(12) #YOY Inflation

macro_ffill["core_cpi_mom"] = macro_ffill["core_cpi"].pct_change(1) # MOM Inflation

macro_ffill["core_cpi_mom3"] = macro_ffill["core_cpi"].pct_change(3) # 3-month momentum

In [4]:
# LAGGED FEATURES

lags = [1, 3, 6, 12]

for col in ["core_cpi_yoy", "core_cpi_mom", "oil_price", "unemployment", "m2", "fed_funds"]:
    for lag in lags:
        macro_ffill[f"{col}_lag{lag}"] = macro_ffill[col].shift(lag)

In [5]:
# DROP NA VALUES

macro_features = macro_ffill.dropna()
macro_features.tail()

,core_cpi,fed_funds,unemployment,oil_price,m2,core_cpi_yoy,core_cpi_mom,core_cpi_mom3,core_cpi_yoy_lag1,core_cpi_yoy_lag3,...,unemployment_lag6,unemployment_lag12,m2_lag1,m2_lag3,m2_lag6,m2_lag12,fed_funds_lag1,fed_funds_lag3,fed_funds_lag6,fed_funds_lag12
date,,,,,,,,,,,,,,,,,,,,,
2026-04-07,334.165,3.64,4.3,114.58,22667.3,0.0,0.0,0.0,0.0,0.0,...,4.3,4.3,22667.3,22667.3,22667.3,22667.3,3.64,3.64,3.64,3.64
2026-04-08,334.165,3.64,4.3,96.17,22667.3,0.0,0.0,0.0,0.0,0.0,...,4.3,4.3,22667.3,22667.3,22667.3,22667.3,3.64,3.64,3.64,3.64
2026-04-09,334.165,3.64,4.3,99.62,22667.3,0.0,0.0,0.0,0.0,0.0,...,4.3,4.3,22667.3,22667.3,22667.3,22667.3,3.64,3.64,3.64,3.64
2026-04-10,334.165,3.64,4.3,98.34,22667.3,0.0,0.0,0.0,0.0,0.0,...,4.3,4.3,22667.3,22667.3,22667.3,22667.3,3.64,3.64,3.64,3.64
2026-04-13,334.165,3.64,4.3,100.72,22667.3,0.0,0.0,0.0,0.0,0.0,...,4.3,4.3,22667.3,22667.3,22667.3,22667.3,3.64,3.64,3.64,3.64


In [6]:
# SAVE PROCESSED FEATURE DATA

macro_features.to_parquet("../data/macro_features.parquet")
print("Saved macro_features.parquet")

Saved macro_features.parquet
